In [1]:
!pip install -q -U transformers accelerate requests

import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
import json
import glob
import time
import zlib
import requests
import numpy as np
import pandas as pd
import torch
from concurrent.futures import ThreadPoolExecutor
from PIL import Image
from transformers import AutoProcessor, Qwen2_5_VLForConditionalGeneration

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

MODEL_NAME = "Qwen/Qwen2.5-VL-7B-Instruct"
processor = AutoProcessor.from_pretrained(MODEL_NAME, min_pixels=256 * 28 * 28, max_pixels=1024 * 28 * 28)
model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map="auto",
    max_memory={0: "12GiB", 1: "12GiB"},
)
model.eval()
for p_ in model.parameters():
    p_.requires_grad_(False)

layers = model.model.language_model.layers
IMAGE_TOKEN_ID = model.config.image_token_id
print(len(layers), "language model layers, image token id", IMAGE_TOKEN_ID)

device: cuda


preprocessor_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

chat_template.json:   0%|          | 0.00/1.05k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.37k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/5.70k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

... (4 outputs left out)

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/729 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/216 [00:00<?, ?B/s]

28 language model layers, image token id 151655


In [2]:
INPUT = "/kaggle/input/datasets/nocturnalnerd18"
USERNAME = "nocturnalnerd18"
AMBER_IMAGE_DIR = "/kaggle/input/datasets/nocturnalnerd18/amber-hallucination/image"
VG_CACHE_DIR = "/kaggle/temp/vg"
VG_URLS = ["https://cs.stanford.edu/people/rak248/VG_100K", "https://cs.stanford.edu/people/rak248/VG_100K_2"]
os.makedirs(VG_CACHE_DIR, exist_ok=True)

NOISE_RUNS = {"noise_1.5": (1.5, ""), "noise_3.0_retest": (3.0, "_retest"), "noise_4.5": (4.5, ""), "noise_6.0": (6.0, "")}
CONDITIONS = ["clean", "gray", "text_only"] + list(NOISE_RUNS)
N_CONTROLS_PER_TYPE = 300
CONTROL_SHARES = {"true_yes": 0.6, "true_no": 0.4}
BATCH_SAVE_SIZE = 100
OUT_DIR = "/kaggle/working/modes_ref_qwen"
os.makedirs(OUT_DIR, exist_ok=True)


def load_image(path):
    return Image.open(path).convert("RGB")


def render_prompt(text, with_image):
    content = ([{"type": "image"}] if with_image else []) + [{"type": "text", "text": text}]
    return processor.apply_chat_template([{"role": "user", "content": content}], add_generation_prompt=True)


def inputs_with_image(image, text):
    return processor(text=render_prompt(text, True), images=image, return_tensors="pt").to(model.device, torch.float16)


def inputs_text_only(text):
    return processor(text=render_prompt(text, False), return_tensors="pt").to(model.device)

VG_INDEX = {}
for root_ in glob.glob("/kaggle/input/**/*isual*enome*", recursive=True):
    if os.path.isdir(root_):
        for dp, dn, fn in os.walk(root_):
            for name in fn:
                if name.endswith(".jpg"):
                    VG_INDEX.setdefault(name, os.path.join(dp, name))
print(len(VG_INDEX), "Visual Genome images indexed from the attached dataset")


def find_vg_image_path(image_id):
    name = f"{image_id}.jpg"
    if name in VG_INDEX:
        return VG_INDEX[name]
    local = os.path.join(VG_CACHE_DIR, name)
    if os.path.exists(local):
        return local
    for attempt in range(3):
        for base in VG_URLS:
            try:
                r = requests.get(f"{base}/{image_id}.jpg", timeout=30)
            except requests.RequestException:
                continue
            if r.status_code == 200 and len(r.content) > 1000:
                with open(local + ".part", "wb") as f:
                    f.write(r.content)
                os.replace(local + ".part", local)
                return local
        time.sleep(2 * (attempt + 1))
    return None


def resolve_image_path(q):
    qid = q["question_id"]
    if qid.startswith("amber_"):
        return os.path.join(AMBER_IMAGE_DIR, f"{q['image_id']}.jpg")
    if qid.startswith("reefknot_"):
        return find_vg_image_path(q["image_id"])
    return None


with open(f"{INPUT}/vlm-questions/all_questions.json") as f:
    all_questions = json.load(f)
assert len(all_questions) == 17492, "attach vlm-questions version 2"
question_by_id = {q["question_id"]: q for q in all_questions}

with open(f"{INPUT}/dla-qwen/dla_meta.json") as f:
    dla_meta = json.load(f)
PAIRS = {k: tuple(v) for k, v in dla_meta["pairs"].items()}
for prefix, (yes_id, no_id) in PAIRS.items():
    words = [processor.tokenizer.decode([i]).strip() for i in (yes_id, no_id)]
    assert [w.lower() for w in words] == ["yes", "no"], words
assert set(PAIRS) == {"amber_", "reefknot_"}, set(PAIRS)

with open(f"{INPUT}/atp-qwen-selection/selection.json") as f:
    selection = json.load(f)
rng = np.random.default_rng(0)
chosen = []
for t, ids_t in selection["selected"].items():
    by_stratum = {s: [q for q in ids_t if selection["stratum"][q] == s] for s in ("false_yes", "true_yes", "true_no")}
    chosen += by_stratum["false_yes"]
    for s, share in CONTROL_SHARES.items():
        pool = by_stratum[s]
        k = min(int(N_CONTROLS_PER_TYPE * share), len(pool))
        chosen += list(rng.choice(pool, size=k, replace=False))
chosen = list(dict.fromkeys(chosen))
print(len(chosen), "questions chosen")
print(pd.crosstab([pd.Series([question_by_id[q]["hallucination_type"] for q in chosen], name="type"), pd.Series([selection["source"][q] for q in chosen], name="source")], pd.Series([selection["stratum"][q] for q in chosen], name="stratum")).to_string())

108249 Visual Genome images indexed from the attached dataset
1719 questions chosen
stratum             false_yes  true_no  true_yes
type      source                                
attribute amber           113      120       180
relation  amber             4       42        31
          reefknot       1002       78       149


In [3]:
STATE = {"noise": None, "seed": 0, "vis": None}


def _layer0_pre(module, args, kwargs):
    if STATE["noise"] is None:
        return None
    h = args[0] if args else kwargs["hidden_states"]
    h = h.clone()
    vis = STATE["vis"]
    x = h[:, vis].float()
    generator = torch.Generator().manual_seed(STATE["seed"])
    noise = torch.randn(x.shape, generator=generator).to(h.device)
    h[:, vis] = (x + noise * x.std() * STATE["noise"]).to(h.dtype)
    if args:
        return (h,) + tuple(args[1:]), kwargs
    kwargs = dict(kwargs)
    kwargs["hidden_states"] = h
    return args, kwargs


_handle = layers[0].register_forward_pre_hook(_layer0_pre, with_kwargs=True)


@torch.no_grad()
def delta_of(inputs, sign, yes_id, no_id, noise=None, seed=0):
    STATE.update(noise=noise, seed=seed, vis=(inputs["input_ids"][0] == IMAGE_TOKEN_ID))
    out = model(**inputs, use_cache=False, logits_to_keep=1)
    STATE["noise"] = None
    logits = out.logits[0, -1].float()
    d = float(sign * (logits[yes_id] - logits[no_id]))
    del out, logits
    return d if np.isfinite(d) else float("nan")

In [4]:
def collect(tag, keep):
    todo_ids = [qid for qid in chosen if keep(qid)]
    paths = sorted(glob.glob(os.path.join(OUT_DIR, f"{tag}_batch_*.pt")))
    done_ids = set()
    for path in paths:
        done_ids.update(e["question_id"] for e in torch.load(path, weights_only=False))
    todo = [qid for qid in todo_ids if qid not in done_ids]
    print(f"{tag}: {len(todo_ids)} questions, {len(done_ids)} already done, {len(todo)} to run")

    batch_entries, batch_index, missing, failed = [], len(paths), 0, 0
    t_start = time.time()

    for i, qid in enumerate(todo):
        q = question_by_id[qid]
        image_path = resolve_image_path(q)
        if image_path is None or not os.path.exists(image_path):
            missing += 1
            continue

        sign = 1.0 if q["ground_truth_answer"].strip().lower() == "yes" else -1.0
        text = q["question_text"]
        yes_id, no_id = PAIRS[qid.split("_")[0] + "_"]
        image = load_image(image_path)
        inp = inputs_with_image(image, text)
        deltas = {"clean": delta_of(inp, sign, yes_id, no_id)}
        for name, (level, suffix) in NOISE_RUNS.items():
            deltas[name] = delta_of(inp, sign, yes_id, no_id, noise=level, seed=zlib.crc32((qid + suffix).encode()) % (2 ** 31))
        del inp
        deltas["gray"] = delta_of(inputs_with_image(Image.new("RGB", image.size, (128, 128, 128)), text), sign, yes_id, no_id)
        deltas["text_only"] = delta_of(inputs_text_only(text), sign, yes_id, no_id)

        entry = {
            "question_id": qid, "hallucination_type": q["hallucination_type"], "stratum": selection["stratum"][qid],
            "group": selection["group"][qid], "source": selection["source"][qid], "split": selection["split"][qid],
            "sign": sign, "ok": bool(np.isfinite(deltas["clean"])), **{f"delta_{k}": v for k, v in deltas.items()},
        }
        failed += int(not entry["ok"])
        batch_entries.append(entry)

        if len(batch_entries) >= BATCH_SAVE_SIZE:
            torch.save(batch_entries, os.path.join(OUT_DIR, f"{tag}_batch_{batch_index}.pt"))
            batch_entries, batch_index = [], batch_index + 1

        if i % 100 == 0 and i > 0:
            torch.cuda.empty_cache()
            avg = (time.time() - t_start) / (i + 1)
            print(f"[{i + 1}/{len(todo)}] {qid} ({avg:.2f} s/question avg, ETA {avg * (len(todo) - i - 1) / 60:.1f} min, failed {failed}, missing images {missing})")

    if batch_entries:
        torch.save(batch_entries, os.path.join(OUT_DIR, f"{tag}_batch_{batch_index}.pt"))
    print(f"done -- {tag}, failed {failed}, missing images {missing}, {(time.time() - t_start) / 60:.1f} min")
    return missing

In [5]:
missing = collect("main", lambda qid: not qid.startswith("reefknot_"))
assert missing == 0, f"{missing} questions have no image"

main: 490 questions, 0 already done, 490 to run


/usr/local/lib/python3.13/dist-packages/transformers/models/qwen2_vl/image_processing_qwen2_vl.py:129: FutureWarning: Passing `min_pixels` and `max_pixels` to a processor call is deprecated and will be removed in v5.23. Pass in `size={'longest_edge': xxx, 'shortest_edge': xxx} to override the target size.`
  warnings.warn(


[101/490] amber_2632 (4.50 s/question avg, ETA 29.2 min, failed 0, missing images 0)
[201/490] amber_5973 (4.54 s/question avg, ETA 21.9 min, failed 0, missing images 0)
[301/490] amber_4308 (4.40 s/question avg, ETA 13.8 min, failed 0, missing images 0)
[401/490] amber_3024 (4.36 s/question avg, ETA 6.5 min, failed 0, missing images 0)
done -- main, failed 0, missing images 0, 35.4 min


In [6]:
reef_ids = [qid for qid in chosen if qid.startswith("reefknot_")]
reef_images = sorted({question_by_id[qid]["image_id"] for qid in reef_ids})
print(len(reef_ids), "reefknot questions on", len(reef_images), "images,", sum(f"{i}.jpg" in VG_INDEX for i in reef_images), "found in the attached dataset")
t0 = time.time()
with ThreadPoolExecutor(max_workers=16) as pool:
    got = list(pool.map(find_vg_image_path, reef_images))
n_missing = sum(g is None for g in got)
print(f"fetched {len(reef_images) - n_missing} images in {time.time() - t0:.0f} s, missing {n_missing}")
assert n_missing == 0, "some Visual Genome images could not be fetched; rerun this cell"

1229 reefknot questions on 1120 images, 1120 found in the attached dataset
fetched 1120 images in 0 s, missing 0


In [7]:
missing = collect("reefknot", lambda qid: qid.startswith("reefknot_"))
assert missing == 0, f"{missing} questions have no image"

reefknot: 1229 questions, 0 already done, 1229 to run
[101/1229] reefknot_2331649_1 (3.16 s/question avg, ETA 59.5 min, failed 0, missing images 0)
[201/1229] reefknot_2318939_1 (3.14 s/question avg, ETA 53.9 min, failed 0, missing images 0)
[301/1229] reefknot_2330615_1 (3.14 s/question avg, ETA 48.6 min, failed 0, missing images 0)
[401/1229] reefknot_2341393_1 (3.16 s/question avg, ETA 43.6 min, failed 0, missing images 0)
[501/1229] reefknot_2325865_1 (3.14 s/question avg, ETA 38.1 min, failed 0, missing images 0)
[601/1229] reefknot_2366568_1 (3.15 s/question avg, ETA 33.0 min, failed 0, missing images 0)
[701/1229] reefknot_2348565_1 (3.17 s/question avg, ETA 27.9 min, failed 0, missing images 0)
[801/1229] reefknot_2364804_1 (3.18 s/question avg, ETA 22.7 min, failed 0, missing images 0)
[901/1229] reefknot_2378549_1 (3.19 s/question avg, ETA 17.4 min, failed 0, missing images 0)
[1001/1229] reefknot_2334484_1 (3.17 s/question avg, ETA 12.0 min, failed 0, missing images 0)
[1101

In [8]:
entries = []
for path in sorted(glob.glob(os.path.join(OUT_DIR, "*_batch_*.pt"))):
    entries.extend(torch.load(path, weights_only=False))
df = pd.DataFrame(entries)
assert df.question_id.is_unique and set(df.question_id) == set(chosen), "collection incomplete or duplicated"
good = df[df.ok]
print(f"{len(df)} entries, ok {len(good)}, failed {len(df) - len(good)}")

ref_dir = f"{INPUT}/atp-qwen-noising"
if os.path.exists(ref_dir):
    ref = {}
    for path in glob.glob(f"{ref_dir}/**/*_batch_*.pt", recursive=True):
        for e in torch.load(path, weights_only=False):
            if e["ok"]:
                ref[e["question_id"]] = e["delta_clean"]
    common = [q for q in good.question_id if q in ref]
    err = max(abs(good.set_index("question_id").loc[q, "delta_clean"] - ref[q]) for q in common)
    print(f"clean logit difference against atp-qwen-noising over {len(common)} questions: max err {err:.4f}")
    assert err < 0.1, "clean runs differ from the earlier collection"

cols = [f"delta_{c}" for c in CONDITIONS]
print("mean delta by stratum and condition (positive favours the true answer):")
print(good.groupby("stratum")[cols].mean().round(2).to_string())
raw = good[cols].mul(good.sign, axis=0)
print("share of questions where the run answers Yes:")
print((raw > 0).mean().rename(lambda c: c.replace("delta_", "")).round(3).to_string())

with open(os.path.join(OUT_DIR, "refs_meta.json"), "w") as f:
    json.dump({"model": MODEL_NAME, "conditions": CONDITIONS, "noise_runs": {k: list(v) for k, v in NOISE_RUNS.items()},
               "gray_image": "original size, 128", "n_controls_per_type": N_CONTROLS_PER_TYPE, "n_ok": len(good), "n_failed": len(df) - len(good)}, f)
df.to_csv(os.path.join(OUT_DIR, "references.csv"), index=False)

with open(os.path.join(OUT_DIR, "dataset-metadata.json"), "w") as f:
    json.dump({
        "title": "modes-references-qwen",
        "id": f"{USERNAME}/modes-references-qwen",
        "subtitle": "Alternative image references for Qwen2.5-VL false-yes mode analysis",
        "description": "Qwen2.5-VL-7B sign-corrected Yes minus No logit gap (delta, positive when the model favours the true answer) at the last position "
                       "for every false_yes question in atp-qwen-selection plus sampled true_yes and true_no controls per type. "
                       "Conditions: clean; gray (uniform grey image of the original size); text_only (no image in the prompt); noise_1.5, noise_4.5 and noise_6.0 "
                       "(Gaussian noise on the visual embeddings at layer 0 with std equal to the level times the embedding std); "
                       "noise_3.0_retest (level 3.0 with a different seed from atp-qwen-noising). One row per question in references.csv.",
        "licenses": [{"name": "CC0-1.0"}],
    }, f)

!kaggle datasets create -p {OUT_DIR}

1719 entries, ok 1719, failed 0
clean logit difference against atp-qwen-noising over 1719 questions: max err 0.0000
mean delta by stratum and condition (positive favours the true answer):
           delta_clean  delta_gray  delta_text_only  delta_noise_1.5  delta_noise_3.0_retest  delta_noise_4.5  delta_noise_6.0
stratum                                                                                                                       
false_yes        -2.90        1.07             0.76            -2.53                    0.24             3.73             5.38
true_no           8.86        3.52             2.59             8.08                    6.14             7.19             7.94
true_yes          4.43       -2.79            -0.93             3.95                   -0.86            -5.33            -6.91
share of questions where the run answers Yes:
clean               0.860
gray                0.124
text_only           0.307
noise_1.5           0.774
noise_3.0_retest    0.378
n